In [7]:
import pandas as pd
import geopandas as gpd
import unicodedata

# ------------------------------------------------------------
# Caminhos dos arquivos
# ------------------------------------------------------------
csv_pontos = r"C:\Users\gabriel.coimbra\Downloads\SURMIS BD.csv"
gpkg_bacias = r"C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\SHP criado\BACIAS_COM_SB18_SEM_SB02_SB17C.gpkg"

saida_excel = r"C:\Users\gabriel.coimbra\Downloads\resumo_economias_por_bacia.xlsx"

# ------------------------------------------------------------
# Função para padronizar texto
# Remove acentos, espaços extras e deixa tudo maiúsculo
# ------------------------------------------------------------
def normalizar_texto(txt):
    if pd.isna(txt):
        return ""
    txt = str(txt).strip().upper()
    txt = unicodedata.normalize("NFKD", txt)
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    return txt


# ------------------------------------------------------------
# 1. Ler CSV de pontos
# ------------------------------------------------------------
df = pd.read_csv(
    csv_pontos,
    encoding="utf-8",
    encoding_errors="replace",
    delimiter=";"
)

print("Arquivo lido com sucesso.")
print(f"Total de registros no CSV original: {len(df)}")

# ------------------------------------------------------------
# 2. Filtrar apenas município de IJUI
# ------------------------------------------------------------
df["municipio_norm"] = df["Nome do município"].apply(normalizar_texto)

df = df[df["municipio_norm"] == "IJUI"].copy()

print(f"Total de pontos após filtro de município IJUI: {len(df)}")

# ------------------------------------------------------------
# 3. Garantir que colunas numéricas estejam como número
# ------------------------------------------------------------
colunas_numericas = [
    "Latitude (graus)",
    "Longitude (graus)",
    "Econ. Residencial"
]

for col in colunas_numericas:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remover pontos sem coordenada
df = df.dropna(subset=["Latitude (graus)", "Longitude (graus)"]).copy()

print(f"Total de pontos após remover coordenadas inválidas: {len(df)}")

# ------------------------------------------------------------
# 4. Criar GeoDataFrame dos pontos
# ------------------------------------------------------------
gdf_pontos = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df["Longitude (graus)"],
        df["Latitude (graus)"]
    ),
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# 5. Ler bacias
# ------------------------------------------------------------
gdf_bacias = gpd.read_file(gpkg_bacias)

if "Nome" not in gdf_bacias.columns:
    raise ValueError("A coluna 'Nome' não foi encontrada no arquivo de bacias.")

# Manter apenas a coluna de nome e geometria
gdf_bacias = gdf_bacias[["Nome", "geometry"]].copy()

# Corrigir geometrias inválidas, se houver
gdf_bacias["geometry"] = gdf_bacias["geometry"].buffer(0)

# Reprojetar pontos para o CRS das bacias, se necessário
if gdf_pontos.crs != gdf_bacias.crs:
    gdf_pontos = gdf_pontos.to_crs(gdf_bacias.crs)

# ------------------------------------------------------------
# 6. Cruzamento espacial
# ------------------------------------------------------------
pontos_bacias = gpd.sjoin(
    gdf_pontos,
    gdf_bacias,
    how="left",
    predicate="within"
)

# ------------------------------------------------------------
# 7. Criar indicadores
# ------------------------------------------------------------

# Econ. Residencial diferente de zero
pontos_bacias["econ_res_dif_zero"] = (
    pontos_bacias["Econ. Residencial"]
    .fillna(0)
    .ne(0)
)

# Normalizar situação da ligação de esgoto
pontos_bacias["situacao_esgoto_norm"] = (
    pontos_bacias["Situação da ligação de esgoto"]
    .apply(normalizar_texto)
)

# Econ. Residencial diferente de zero E situação da ligação de esgoto = Ligado
pontos_bacias["econ_res_dif_zero_ligado"] = (
    pontos_bacias["econ_res_dif_zero"]
    & pontos_bacias["situacao_esgoto_norm"].eq("LIGADO")
)

# Valor de Econ. Residencial apenas para situação = Ligado
pontos_bacias["econ_residencial_ligado"] = pontos_bacias["Econ. Residencial"].where(
    pontos_bacias["situacao_esgoto_norm"].eq("LIGADO"),
    0
)

# Valor de Econ. Residencial apenas para situação = Ligado e diferente de zero
pontos_bacias["econ_residencial_dif_zero_ligado"] = pontos_bacias["Econ. Residencial"].where(
    pontos_bacias["econ_res_dif_zero_ligado"],
    0
)

# ------------------------------------------------------------
# 8. Resumo por bacia
# ------------------------------------------------------------
resumo = (
    pontos_bacias
    .groupby("Nome", dropna=False)
    .agg(
        qtd_pontos_econ_res_dif_zero=("econ_res_dif_zero", "sum"),
        soma_econ_residencial=("Econ. Residencial", "sum"),
        qtd_pontos_econ_res_dif_zero_ligado=("econ_res_dif_zero_ligado", "sum"),
        soma_econ_residencial_ligado=("econ_residencial_ligado", "sum"),
        soma_econ_residencial_dif_zero_ligado=("econ_residencial_dif_zero_ligado", "sum"),
        qtd_total_pontos=("Econ. Residencial", "count")
    )
    .reset_index()
)

# Renomear pontos fora de bacia
resumo["Nome"] = resumo["Nome"].fillna("Fora das bacias")

# Ordenar
resumo = resumo.sort_values("Nome").reset_index(drop=True)

# ------------------------------------------------------------
# 9. Exportar resultados
# ------------------------------------------------------------
resumo.to_excel(saida_excel, index=False)

# ------------------------------------------------------------
# 10. Exibir resultado
# ------------------------------------------------------------
print("\nResumo por bacia:")
print(resumo)

print(f"\nResumo exportado para: {saida_excel}")

Arquivo lido com sucesso.
Total de registros no CSV original: 262514
Total de pontos após filtro de município IJUI: 26449
Total de pontos após remover coordenadas inválidas: 26433

Resumo por bacia:
               Nome  qtd_pontos_econ_res_dif_zero  soma_econ_residencial  \
0   Fora das bacias                           441                    468   
1             SB-01                          2158                   2569   
2             SB-02                            23                     35   
3             SB-05                           581                   1330   
4             SB-07                          1102                   1786   
5             SB-08                           962                   2095   
6             SB-09                           354                    682   
7             SB-0A                           318                    493   
8             SB-0B                           479                    894   
9             SB-0C                      